In [1]:
# Load with Level 3 (most granular usable level)
from datasets import load_dataset

print("Loading Level 1 (default - 21 labels)...")
dataset_l1 = load_dataset('coastalcph/multi_eurlex', 'en', split='train[:10]', trust_remote_code=True)
classlabel_l1 = dataset_l1.features["labels"].feature
print(f"Level 1 labels: {len(classlabel_l1.names)}")

print("\nLoading Level 2...")
dataset_l2 = load_dataset('coastalcph/multi_eurlex', 'en', split='train[:10]', 
                          label_level='level_2', trust_remote_code=True)
classlabel_l2 = dataset_l2.features["labels"].feature
print(f"Level 2 labels: {len(classlabel_l2.names)}")

print("\nLoading Level 3...")
dataset_l3 = load_dataset('coastalcph/multi_eurlex', 'en', split='train[:10]', 
                          label_level='level_3', trust_remote_code=True)
classlabel_l3 = dataset_l3.features["labels"].feature
print(f"Level 3 labels: {len(classlabel_l3.names)}")

print("\nLoading ALL levels (original labels)...")
dataset_all = load_dataset('coastalcph/multi_eurlex', 'en', split='train[:10]', 
                           label_level='all_levels', trust_remote_code=True)
classlabel_all = dataset_all.features["labels"].feature
print(f"All levels labels: {len(classlabel_all.names)}")

Loading Level 1 (default - 21 labels)...
Level 1 labels: 21

Loading Level 2...


Generating train split:   0%|          | 0/55000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/5000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/5000 [00:00<?, ? examples/s]

Level 2 labels: 127

Loading Level 3...


...

Level 3 labels: 567

Loading ALL levels (original labels)...


Generating train split:   0%|          | 0/55000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/5000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/5000 [00:00<?, ? examples/s]

All levels labels: 7390


In [2]:
# Compare the same document across levels
doc_idx = 0

print("="*80)
print(f"DOCUMENT {doc_idx} - LABELS AT DIFFERENT GRANULARITIES")
print("="*80)

print("\nLevel 1 (21 labels):")
doc_l1 = dataset_l1[doc_idx]
for label_idx in doc_l1['labels']:
    print(f"  {classlabel_l1.int2str(label_idx)}")

print("\nLevel 2 (~127 labels):")
doc_l2 = dataset_l2[doc_idx]
for label_idx in doc_l2['labels']:
    print(f"  {classlabel_l2.int2str(label_idx)}")

print("\nLevel 3 (~500+ labels):")
doc_l3 = dataset_l3[doc_idx]
for label_idx in doc_l3['labels']:
    print(f"  {classlabel_l3.int2str(label_idx)}")

print("\nAll levels (original):")
doc_all = dataset_all[doc_idx]
for label_idx in doc_all['labels']:
    print(f"  {classlabel_all.int2str(label_idx)}")

DOCUMENT 0 - LABELS AT DIFFERENT GRANULARITIES

Level 1 (21 labels):
  100160
  100155
  100158
  100147
  100149

Level 2 (~127 labels):
  100273
  100216
  100261
  100274
  100195
  100242

Level 3 (~500+ labels):
  1386
  2825
  138
  2475
  3879
  3641

All levels (original):
  1706
  1826
  2754
  3690
  4036
  5234
  5309


In [3]:
# Check if there's documentation about label levels in the script
with open('multi_eurlex_script.py', 'r') as f:
    content = f.read()

# Look for the _generate_examples function which processes the data
if '_generate_examples' in content:
    start = content.find('def _generate_examples')
    end = content.find('\n    def ', start + 1)
    if end == -1:
        end = len(content)
    
    print("_generate_examples function:")
    print(content[start:start+2000])  # First 2000 chars

_generate_examples function:
def _generate_examples(self, archive, filepath):
        """This function returns the examples in the raw (text) form."""
        for path, f in archive:
            if path == filepath:
                if self.config.language == "all_languages":
                    for id_, row in enumerate(f):
                        data = json.loads(row)
                        yield id_, {
                            "celex_id": data["celex_id"],
                            "text": {lang: data["text"][lang] for lang in self.config.languages},
                            "labels": data["eurovoc_concepts"][self.config.label_level],
                        }
                else:
                    for id_, row in enumerate(f):
                        data = json.loads(row)
                        if data["text"][self.config.language] is not None:
                            yield id_, {
                                "celex_id": data["celex_id"],
                      